# 03 - Pensioni, demografia e lavoro

Notebook esplorativo sulle tabelle finali principali. Le celle sono costruite per funzionare anche quando i dati non sono ancora popolati.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR = REPO_ROOT / 'code'
DATA_FINAL_DIR = REPO_ROOT / 'data' / 'final'

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## Parametri modificabili

Cambiare gli indicatori sotto se si vogliono esplorare altre serie.

In [ ]:
INDICATORE_SPESA_PIL = 'spesa_pensionistica_pil'
INDICATORE_PENSIONI = 'pensioni_vigenti'
INDICATORE_PENSIONATI = 'pensionati'
INDICATORE_OCCUPATI = 'occupati'
INDICATORE_RAPPORTO = 'pensionati_su_occupati'

In [ ]:
def read_csv_optional(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def indicator(df: pd.DataFrame, indicatore_id: str) -> pd.DataFrame:
    if df.empty or 'indicatore_id' not in df.columns:
        return pd.DataFrame()
    return df[df['indicatore_id'].astype(str).eq(indicatore_id)].copy()


def plot_line(df: pd.DataFrame, title: str, ylabel: str) -> None:
    if df.empty:
        print('Dati non disponibili:', title)
        return
    data = df.copy()
    data['anno'] = pd.to_numeric(data['anno'], errors='coerce')
    data['valore'] = pd.to_numeric(data['valore'], errors='coerce')
    data = data.dropna(subset=['anno', 'valore']).sort_values('anno')
    if data.empty:
        print('Nessun valore numerico:', title)
        return
    plt.figure(figsize=(10, 6))
    plt.plot(data['anno'], data['valore'], marker='o')
    plt.title(title)
    plt.xlabel('Anno')
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

## Caricamento tabelle finali

La pipeline crea le tabelle in `data/final/`. Se le righe sono zero, bisogna completare download e trasformazioni.

In [ ]:
annuale = read_csv_optional(DATA_FINAL_DIR / 'tabella_annuale_pensioni.csv')
demografia = read_csv_optional(DATA_FINAL_DIR / 'tabella_demografia_lavoro.csv')
distribuzione = read_csv_optional(DATA_FINAL_DIR / 'tabella_distribuzione_pensionati.csv')

display(pd.DataFrame({
    'tabella': ['annuale', 'demografia_lavoro', 'distribuzione_pensionati'],
    'righe': [len(annuale), len(demografia), len(distribuzione)],
}))

## Indicatori principali

Questi grafici si attivano solo quando gli indicatori sono presenti nelle tabelle finali.

In [ ]:
plot_line(indicator(annuale, INDICATORE_SPESA_PIL), 'Spesa pensionistica in rapporto al PIL', 'Percentuale del PIL')
plot_line(indicator(demografia, INDICATORE_OCCUPATI), 'Occupati', 'Numero')
plot_line(indicator(demografia, INDICATORE_RAPPORTO), 'Pensionati su occupati', 'Rapporto')

## Pensioni e pensionati

Questa sezione distingue trattamenti e persone. Quando entrambi gli indicatori sono presenti, calcola il rapporto tra trattamenti e beneficiari.

In [ ]:
pensioni = indicator(annuale, INDICATORE_PENSIONI)
pensionati = indicator(annuale, INDICATORE_PENSIONATI)

if pensioni.empty or pensionati.empty:
    print('Servono pensioni_vigenti e pensionati in tabella_annuale_pensioni.csv.')
else:
    rapporto = pensioni[['anno', 'valore']].rename(columns={'valore': 'pensioni'}).merge(
        pensionati[['anno', 'valore']].rename(columns={'valore': 'pensionati'}),
        on='anno',
        how='inner',
    )
    rapporto['valore'] = pd.to_numeric(rapporto['pensioni'], errors='coerce') / pd.to_numeric(rapporto['pensionati'], errors='coerce')
    display(rapporto)
    plot_line(rapporto, 'Trattamenti per beneficiario', 'Rapporto')

## Distribuzione

Quando la tabella distribuzione sara' popolata, qui si potranno leggere classi di importo, eta', sesso e territorio.

In [ ]:
if distribuzione.empty:
    print('tabella_distribuzione_pensionati.csv non disponibile o vuota.')
else:
    display(distribuzione.head(50))